# Step 10b – Round 2 ASR training: Whisper-small

Step 10 (turbo)-এর মতোই, শুধু মডেল **Whisper-small**। Round 1-এর Step 2e-এর সঙ্গে তুলনা করা হবে।

- Training: A1, A4, A6, A7 (Round 1) + B01, B03, B05, B06, B07, B08, B10, B11, B12 (Round 2, শুধু train-ভাগের বাক্য)
- Validation: Round 1-এর হুবহু একই ৩০০টা clip
- Test: `test_seen`, `test_unseen` (A2), `test_new` (B02, B04, B09, প্রধান), `test_new_all` (বাড়তি), `test_r2_seen`
- তুলনা: Round 1-এর Whisper-small (`whisper-small-4spk/final`) একই test set-এ
- শেষে paired bootstrap (Cell 8) আর Drive-এ সেভ (Cell 9)

**চালানোর নিয়ম:** Runtime → Change runtime type → GPU (T4 / L4 / A100) → **Run all**।
মাঝপথে কেটে গেলে আবার Run all দিন, শেষ checkpoint থেকে চলবে। training শেষ হয়ে গেলে আবার চালাতে হলে Cell 6-এ `TRAIN = False`।
কোনো ফাইল মোছা হয় না।


In [ ]:
# Cell 1: Drive (আগে ভুল করে তৈরি লোকাল ফোল্ডার থাকলে নাম বদলে সরিয়ে রাখে, মোছে না)
from google.colab import drive
import os, time
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive') and os.listdir('/content/drive'):
    os.rename('/content/drive', '/content/drive_local_leftover_' + time.strftime('%H%M%S'))
drive.mount('/content/drive')


In [ ]:
# Cell 2: সেটিং + library
MODEL = "small"      # এই notebook: Whisper-small
QUICK = False        # True = ছোট পরীক্ষা (~১৫ মিনিট)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.3", "accelerate==1.3.0",
                "jiwer==3.0.5", "soundfile", "bitsandbytes"], check=True)
import torch, shutil
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported() and 'T4' not in GPU_NAME
print(f"GPU: {GPU_NAME} ({GPU_GB:.0f} GB) | bf16: {BF16} | MODEL: {MODEL} | QUICK: {QUICK}")
if MODEL == "turbo" and GPU_GB < 20:
    print("সতর্কতা: turbo-র জন্য L4/A100 নিন, T4-এ খুব ধীর হবে।")

In [ ]:
# Cell 3: Round 1 + Round 2 WAV (Drive-এর .tar থেকে Colab-এ খোলা)
import tarfile, glob, re
import pandas as pd
ROOT = "/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data"
ASR_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr"
OUT1 = "/content/wav_16k"            # Round 1
OUT2 = "/content/round2_wav"         # Round 2
TRAIN_SPK1 = ["A1", "A4", "A6", "A7"]
UNSEEN_SPK = ["A2"]
TRAIN_SPK2 = ["B01", "B03", "B05", "B06", "B07", "B08", "B10", "B11", "B12"]
NEW_SPK = ["B02", "B04", "B09"]
os.makedirs(OUT1, exist_ok=True); os.makedirs(OUT2, exist_ok=True)

# ---- Round 1: Step 2d-এর হুবহু একই কোড ----
KEEP = TRAIN_SPK1 + UNSEEN_SPK
meta = pd.read_csv(ASR_DIR + "/metadata.csv")
meta = meta[meta.speaker.isin(KEEP)].copy()
meta["audio_path"] = OUT1 + "/" + meta.audio_path.str.split("/wav_16k/").str[-1]
meta["folder"] = meta.audio_path.str[len(OUT1) + 1:].str.split("/").str[0]
for t in sorted(glob.glob(ROOT + "/wav_16k_tar/*.tar")):
    s = os.path.basename(t)[:-4]
    if re.fullmatch(r"A\d+", s) and s not in KEEP:
        continue
    want = meta.audio_path[meta.folder == s]
    if len(want) > 0 and want.map(os.path.exists).all():
        continue
    print("extracting R1", s)
    with tarfile.open(t) as tf:
        tf.extractall(OUT1)

# ---- Round 2 ----
for t in sorted(glob.glob(ROOT + "/round2/wav_16k_tar/*.tar")):
    s = os.path.basename(t)[:-4]
    if os.path.isdir(f"{OUT2}/{s}"):
        continue
    print("extracting R2", s)
    with tarfile.open(t) as tf:
        tf.extractall(OUT2)
r2 = pd.read_csv(ROOT + "/round2/recheck/round2_clean.csv")
r2["audio_path"] = OUT2 + "/" + r2["path"].str.replace("wav_16k/", "", regex=False)
r2 = r2.rename(columns={"sylheti": "text", "duration_s": "duration_sec"})
print("R1 wav:", int(meta.audio_path.map(os.path.exists).sum()), "/", len(meta),
      "| R2 wav:", int(r2.audio_path.map(os.path.exists).sum()), "/", len(r2))
print(f"Drive: {shutil.disk_usage('/content/drive').free/1e9:.0f} GB খালি")

In [ ]:
# Cell 4: train / validation / test
ok = meta.audio_path.map(os.path.exists)
meta = meta[ok & (meta.duration_sec <= 30) & meta.text.notna()
            & meta.split.isin(["train", "validation", "test_seen", "test_unseen"])].reset_index(drop=True)
sp = {k: g.reset_index(drop=True) for k, g in meta.groupby("split")}
train1, val_df = sp["train"], sp["validation"]
val_small = val_df.sample(min(300, len(val_df)), random_state=13).reset_index(drop=True)   # Round 1-এর হুবহু একই ৩০০টা

r2 = r2[r2.audio_path.map(os.path.exists) & (r2.duration_sec <= 30) & r2.text.notna()].copy()
cols = ["audio_path", "speaker", "excel_row", "text", "duration_sec"]
train2 = r2[r2.speaker.isin(TRAIN_SPK2) & (r2.split == "train")][cols]
sp["test_new"] = r2[r2.speaker.isin(NEW_SPK) & (r2.split == "test")][cols].reset_index(drop=True)
sp["test_new_all"] = r2[r2.speaker.isin(NEW_SPK)][cols].reset_index(drop=True)
sp["test_r2_seen"] = r2[r2.speaker.isin(TRAIN_SPK2) & (r2.split == "test")][cols].reset_index(drop=True)

# নিরাপত্তা পরীক্ষা: কোনো test বাক্য training-এ নেই তো?
test_rows = (set(sp["test_seen"].excel_row) | set(sp["test_unseen"].excel_row)
             | set(sp["test_new"].excel_row) | set(sp["test_r2_seen"].excel_row))
leak = set(train1.excel_row) | set(train2.excel_row)
print("test বাক্য training-এ:", len(test_rows & leak), "(০ হওয়া উচিত)")
assert len(test_rows & leak) == 0, "test বাক্য training-এ ঢুকে গেছে!"
assert not set(train2.speaker) & set(NEW_SPK)

train_df = pd.concat([train1[cols], train2], ignore_index=True)
if QUICK:
    train_df = train_df.sample(400, random_state=13).reset_index(drop=True)
    val_small = val_small.head(50)
print(f"train: {len(train_df)} (Round 1: {len(train1)}, Round 2: {len(train2)}) | "
      f"{train_df.duration_sec.sum()/3600:.1f} h | validation: {len(val_small)}")
for k in ["test_seen", "test_unseen", "test_new", "test_new_all", "test_r2_seen"]:
    print(f"{k}: {len(sp[k])} clips, speakers {sorted(sp[k].speaker.unique())}")

In [ ]:
# Cell 5: model + data loader (Step 2d/2e-এর মতোই)
import unicodedata, numpy as np, soundfile as sf, jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
BASE = "openai/whisper-large-v3-turbo" if MODEL == "turbo" else "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(BASE, language="bengali", task="transcribe")
tok, fe = processor.tokenizer, processor.feature_extractor

ZW = dict.fromkeys(map(ord, "\u200b\u200c\ufeff\u00ad\u2060"), None)
def clean(s):
    s = unicodedata.normalize("NFC", str(s)).translate(ZW)
    s = re.sub("\u200d(?!্য)", "", s)
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

class SpeechSet(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, sr = sf.read(r.audio_path, dtype="float32")
        if a.ndim > 1: a = a.mean(1)
        feats = fe(a[: 30 * 16000], sampling_rate=16000).input_features[0]
        labels = tok(clean(r.text)).input_ids[:448]
        return {"input_features": feats, "labels": labels}

def collate(batch):
    x = fe.pad([{"input_features": b["input_features"]} for b in batch], return_tensors="pt")
    y = tok.pad([{"input_ids": b["labels"]} for b in batch], return_tensors="pt")
    labels = y.input_ids.masked_fill(y.attention_mask.ne(1), -100)
    if (labels[:, 0] == model.config.decoder_start_token_id).all():
        labels = labels[:, 1:]
    x["labels"] = labels
    return x

def setup(m):
    m.generation_config.language = "bengali"
    m.generation_config.task = "transcribe"
    m.generation_config.forced_decoder_ids = None
    m.config.forced_decoder_ids = None
    return m

model = setup(WhisperForConditionalGeneration.from_pretrained(BASE))
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.use_cache = False

def compute_metrics(p):
    pred, lab = p.predictions, p.label_ids
    lab = np.where(lab == -100, tok.pad_token_id, lab)
    hyp = [clean(t) for t in tok.batch_decode(pred, skip_special_tokens=True)]
    ref = [clean(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
    pairs = [(r, h) for r, h in zip(ref, hyp) if r]
    ref, hyp = [r for r, _ in pairs], [h if h else "-" for _, h in pairs]
    return {"wer": 100 * jiwer.wer(ref, hyp), "cer": 100 * jiwer.cer(ref, hyp)}
print("ready:", BASE, f"({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

In [ ]:
# Cell 6: training (checkpoint Drive-এ, resume হয়)
TRAIN = True          # False = training বাদ, সেভ করা final মডেল দিয়ে শুধু test
import json, gc
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
NAME = ("whisper-large-v3-turbo" if MODEL == "turbo" else "whisper-small") + "-round2"
RUN = ASR_DIR + "/" + NAME + ("-quick" if QUICK else "")
FINAL = RUN + "/final"
os.makedirs(RUN, exist_ok=True)

# অসম্পূর্ণ checkpoint-এর নাম বদলানো (মুছছে না)
for d in glob.glob(RUN + "/checkpoint-*"):
    if not os.path.exists(d + "/trainer_state.json"):
        os.rename(d, d.replace("/checkpoint-", "/incomplete-checkpoint-"))
        print("renamed incomplete:", os.path.basename(d))

class StopIfNoBetterWER(TrainerCallback):
    def __init__(self, wait): self.wait = wait
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        wer = (metrics or {}).get("eval_wer")
        if wer is None or state.best_metric is None or wer < state.best_metric or not state.best_model_checkpoint:
            return
        best_step = int(str(state.best_model_checkpoint).rstrip("/").split("-")[-1])
        if state.global_step - best_step >= self.wait:
            print(f"No new best WER for {self.wait} steps (best {state.best_metric:.2f} at {best_step}) -> stopping")
            control.should_training_stop = True

if MODEL == "turbo":
    BIG = GPU_GB >= 20
    bs, acc, ebs = (8, 4, 16) if BIG else (4, 8, 8)       # Step 2d: effective batch 32
else:
    bs, acc, ebs = 16, 2, 16                              # Step 2e: effective batch 32

if TRAIN:
    args = Seq2SeqTrainingArguments(
        output_dir=RUN,
        per_device_train_batch_size=bs, gradient_accumulation_steps=acc, per_device_eval_batch_size=ebs,
        learning_rate=1e-5, warmup_steps=20 if QUICK else 300,
        num_train_epochs=5, max_steps=20 if QUICK else -1,
        bf16=BF16, fp16=not BF16,
        optim="adamw_bnb_8bit", gradient_checkpointing=True,
        eval_strategy="steps", save_strategy="steps",
        eval_steps=10 if QUICK else 250, save_steps=10 if QUICK else 250,
        save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="wer", greater_is_better=False,
        predict_with_generate=True, generation_max_length=225,
        logging_steps=5 if QUICK else 50, report_to="none",
        dataloader_num_workers=2, remove_unused_columns=False,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=SpeechSet(train_df), eval_dataset=SpeechSet(val_small),
                             data_collator=collate, compute_metrics=compute_metrics,
                             callbacks=[StopIfNoBetterWER(wait=20 if QUICK else 1000)])
    last = get_last_checkpoint(RUN)
    print("resuming from", last if last else "the start")
    trainer.train(resume_from_checkpoint=last)
    best = trainer.state.best_model_checkpoint
    print("best checkpoint:", best, "| best validation WER:", trainer.state.best_metric)
    trainer.save_model(FINAL); processor.save_pretrained(FINAL)
    json.dump({"best_checkpoint": best, "best_val_wer": trainer.state.best_metric,
               "steps": trainer.state.global_step, "epochs": round(trainer.state.epoch, 2),
               "train_clips": len(train_df), "train_speakers": TRAIN_SPK1 + TRAIN_SPK2,
               "unseen_speakers": UNSEEN_SPK, "new_speakers": NEW_SPK},
              open(RUN + "/train_info.json", "w"), indent=1)
    del trainer
print("final model ->", FINAL)
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 7: test — Round 2 মডেল (Round 1 + Round 2 data) বনাম Round 1 মডেল, একই test set-এ
# প্রতিটা test set শেষ হলেই ফল Drive-এ সেভ হয়; মাঝপথে থামলে আবার চালালে বাকি অংশ থেকে শুরু করে
RERUN_TEST = False
from tqdm.auto import tqdm
DT = torch.bfloat16 if BF16 else torch.float16
R1_FINAL = ASR_DIR + ("/whisper-large-v3-turbo-4spk/final" if MODEL == "turbo" else "/whisper-small-4spk/final")
TEST_SETS = [k for k in ["test_seen", "test_unseen", "test_new", "test_new_all", "test_r2_seen"] if k in sp]
assert os.path.exists(FINAL + "/config.json"), "Round 2 মডেল পাওয়া যায়নি (Drive যুক্ত আছে তো?): " + FINAL

def transcribe(m, df, bs=16):
    m.eval(); m.config.use_cache = True; out = []
    for i in tqdm(range(0, len(df), bs), leave=False):
        b = [SpeechSet(df.iloc[i:i + bs].reset_index(drop=True))[j] for j in range(min(bs, len(df) - i))]
        x = fe.pad([{"input_features": e["input_features"]} for e in b], return_tensors="pt")
        with torch.no_grad(), torch.autocast("cuda", dtype=DT):
            ids = m.generate(x.input_features.to("cuda"), max_new_tokens=440)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out

def score(df):
    ref, hyp = df.text.map(clean).tolist(), [clean(h) or "-" for h in df.pred]
    return round(100 * jiwer.wer(ref, hyp), 1), round(100 * jiwer.cer(ref, hyp), 1)

RES, RES_SPK = RUN + "/results.csv", RUN + "/results_per_speaker.csv"
old = pd.read_csv(RES) if os.path.exists(RES) and not RERUN_TEST else pd.DataFrame(columns=["model", "test set"])
old_spk = pd.read_csv(RES_SPK) if os.path.exists(RES_SPK) and not RERUN_TEST else pd.DataFrame()
rows, per_spk = old.to_dict("records"), old_spk.to_dict("records")
done = set(zip(old["model"], old["test set"]))

systems = {"Round 2 model": FINAL}
if os.path.exists(R1_FINAL + "/config.json"): systems["Round 1 model"] = R1_FINAL
else: print("সতর্কতা: Round 1 মডেল পাওয়া যায়নি:", R1_FINAL)

for sys_name, path in systems.items():
    todo = [n for n in TEST_SETS if (sys_name, n) not in done]
    if not todo:
        print(sys_name, ": সব test আগেই হয়ে গেছে"); continue
    try: del model
    except NameError: pass
    gc.collect(); torch.cuda.empty_cache()
    model = setup(WhisperForConditionalGeneration.from_pretrained(path, torch_dtype=DT)).to("cuda")
    tag = "r2" if path == FINAL else "r1"
    for name in todo:
        df = sp[name].head(100) if QUICK else sp[name]
        print(f"{sys_name} | {name} | {len(df)} clips")
        df = df.assign(pred=transcribe(model, df))
        w, c = score(df)
        print(f"   WER {w} | CER {c}")
        rows.append({"model": sys_name, "test set": name, "files": len(df), "WER": w, "CER": c})
        for s, g in df.groupby("speaker"):
            per_spk.append({"model": sys_name, "test set": name, "speaker": s, "files": len(g),
                            **dict(zip(["WER", "CER"], score(g)))})
        df[["audio_path", "speaker", "excel_row", "text", "pred"]].to_csv(
            f"{RUN}/predictions_{tag}_{name}.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(rows).to_csv(RES, index=False)          # প্রতিটা set শেষে সেভ
        pd.DataFrame(per_spk).to_csv(RES_SPK, index=False)

res = pd.DataFrame(rows)
print("\n==== ফলাফল (WER / CER %) ====")
print(res.pivot_table(index="test set", columns="model", values=["WER", "CER"]).round(1).to_string())
print("\n==== বক্তা অনুযায়ী (Round 2 model) ====")
ps = pd.DataFrame(per_spk)
print(ps[ps["model"] == "Round 2 model"].pivot_table(index=["test set", "speaker"], values=["WER", "CER"]).round(1).to_string())
print("\nsaved:", RUN)


In [ ]:
# Cell 8: paired bootstrap — Round 2 মডেলের উন্নতি কি কাকতালীয়? (Cell 7-এর predictions ফাইল থেকে)
B = 10000
rng = np.random.default_rng(13)

def per_clip(refs, hyps, chars=False):
    """প্রতিটা ক্লিপের ভুলের সংখ্যা আর রেফারেন্সের দৈর্ঘ্য"""
    err, n = [], []
    for r, h in zip(refs, hyps):
        o = jiwer.process_characters(r, h) if chars else jiwer.process_words(r, h)
        err.append(o.substitutions + o.deletions + o.insertions)
        n.append(o.substitutions + o.deletions + o.hits)
    return np.array(err), np.array(n)

brows = []
for name in TEST_SETS:
    f1, f2 = f"{RUN}/predictions_r1_{name}.csv", f"{RUN}/predictions_r2_{name}.csv"
    if not (os.path.exists(f1) and os.path.exists(f2)):
        print("পাওয়া যায়নি:", name); continue
    d1, d2 = pd.read_csv(f1), pd.read_csv(f2)
    d = d1.merge(d2[["audio_path", "pred"]], on="audio_path", suffixes=("_r1", "_r2"))
    d["ref"] = d.text.map(clean)
    d = d[d.ref != ""].reset_index(drop=True)
    for metric, chars in [("WER", False), ("CER", True)]:
        e1, n = per_clip(d.ref, [clean(h) or "-" for h in d.pred_r1], chars)
        e2, _ = per_clip(d.ref, [clean(h) or "-" for h in d.pred_r2], chars)
        idx = rng.integers(0, len(d), size=(B, len(d)))          # প্রতিবার একই ক্লিপ দুটো মডেলের জন্য
        diff = 100 * (e2[idx].sum(1) - e1[idx].sum(1)) / n[idx].sum(1)   # ঋণাত্মক = Round 2 ভালো
        lo, hi = np.percentile(diff, [2.5, 97.5])
        p = min(1.0, 2 * min((diff >= 0).mean(), (diff <= 0).mean()))
        brows.append({"test set": name, "metric": metric, "clips": len(d),
                      "Round 1": round(100 * e1.sum() / n.sum(), 1),
                      "Round 2": round(100 * e2.sum() / n.sum(), 1),
                      "difference": round(100 * (e2.sum() - e1.sum()) / n.sum(), 1),
                      "95% CI": f"[{lo:.1f}, {hi:.1f}]",
                      "Round 2 better (% of resamples)": round(100 * (diff < 0).mean(), 1),
                      "p": "<0.0001" if p == 0 else round(p, 4)})

bout = pd.DataFrame(brows)
print(bout.to_string(index=False))
bout.to_csv(RUN + "/bootstrap_r1_vs_r2.csv", index=False)
print("\nsaved:", RUN + "/bootstrap_r1_vs_r2.csv")


In [ ]:
# Cell 9: সব ফাইল Drive-এ পাঠানো শেষ হওয়া পর্যন্ত অপেক্ষা (কিছু মোছে না)
from google.colab import drive
drive.flush_and_unmount()
print('সব ফাইল Drive-এ সেভ হয়েছে। এখন Runtime → Disconnect and delete runtime দিতে পারেন।')